In [20]:
import pandas as pd
from collections import defaultdict

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
INPUT_PATH  = "webnlg_qa_merged_validated.csv"   # output from the validation step
OUTPUT_PATH = "coverage_report.csv"

df = pd.read_csv(INPUT_PATH)

# Use corrected_answer to determine yes/no polarity
# Fall back to original answer if corrected_answer is missing
df["effective_answer"] = df["corrected_answer"].fillna(df["answer"]).astype(str).str.strip()

# ─────────────────────────────────────────────────────────────
# GROUP BY (category, eid, lex_id)
# ─────────────────────────────────────────────────────────────
GROUP_KEYS = ["category", "eid", "lex_id"]

records = []

for group_key, group in df.groupby(GROUP_KEYS):
    cat, eid, lex_id = group_key

    # ── Represent the lexicalisation text ──
    lex_text = group["lex_text"].iloc[0]
    statement = group["statement"].iloc[0] if "statement" in group.columns else ""

    # ── Collect questions by type ──
    yes_no_rows   = group[group["question_type"] == "yes_no"]
    extr_rows     = group[group["question_type"] == "extractive"]

    # Yes/No split using effective (corrected) answer
    yes_questions = yes_no_rows[yes_no_rows["effective_answer"].str.lower() == "yes"]["question"].unique().tolist()
    no_questions  = yes_no_rows[yes_no_rows["effective_answer"].str.lower() == "no"]["question"].unique().tolist()
    extr_questions = extr_rows["question"].unique().tolist()

    # ── Coverage booleans ──
    has_yes       = len(yes_questions) >= 1
    has_no        = len(no_questions)  >= 1
    has_2_extr    = len(extr_questions) >= 1

    is_complete   = has_yes and has_no and has_2_extr

    # ── Missing flags ──
    missing = []
    if not has_yes:
        missing.append("yes_question")
    if not has_no:
        missing.append("no_question")
    if not has_2_extr:
        n_missing_extr = 2 - len(extr_questions)
        missing.append(f"extractive_questions({n_missing_extr}_missing)")

    records.append({
        "category":           cat,
        "eid":                eid,
        "lex_id":             lex_id,
        "lex_text":           lex_text,
        "statement":          statement,
        # Counts
        "n_yes_questions":    len(yes_questions),
        "n_no_questions":     len(no_questions),
        "n_extractive_questions": len(extr_questions),
        # Coverage
        "has_yes":            has_yes,
        "has_no":             has_no,
        "has_2_extractive":   has_2_extr,
        "is_complete":        is_complete,
        # Detail
        "missing":            "; ".join(missing) if missing else "",
        "yes_questions":      " | ".join(yes_questions),
        "no_questions":       " | ".join(no_questions),
        "extractive_questions": " | ".join(extr_questions),
    })

report_df = pd.DataFrame(records)

# ─────────────────────────────────────────────────────────────
# SUMMARY STATS
# ─────────────────────────────────────────────────────────────
total       = len(report_df)
complete    = report_df["is_complete"].sum()
incomplete  = total - complete

missing_counts = {
    "missing_yes":        (~report_df["has_yes"]).sum(),
    "missing_no":         (~report_df["has_no"]).sum(),
    "missing_extractive": (~report_df["has_2_extractive"]).sum(),
}

# Break down combos of what's missing
missing_combo = report_df[~report_df["is_complete"]].groupby("missing").size().reset_index(name="count")

print("=" * 60)
print(f"COVERAGE REPORT — grouped by (category, eid, lex_id)")
print("=" * 60)
print(f"  Total lexicalisations:    {total}")
print(f"  ✅ Complete:              {complete}  ({100*complete/total:.1f}%)")
print(f"  ❌ Incomplete:            {incomplete}  ({100*incomplete/total:.1f}%)")
print()
print("Missing breakdown (lexicalisations missing each type):")
print(f"  • No YES question:        {missing_counts['missing_yes']}")
print(f"  • No NO question:         {missing_counts['missing_no']}")
print(f"  • <2 extractive:          {missing_counts['missing_extractive']}")
print()
print("Missing combination breakdown:")
print(missing_combo.to_string(index=False))
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────
report_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nFull report saved to: {OUTPUT_PATH}")

# ─────────────────────────────────────────────────────────────
# OPTIONAL: show sample incomplete rows
# ─────────────────────────────────────────────────────────────
incomplete_df = report_df[~report_df["is_complete"]]
if len(incomplete_df) > 0:
    print(f"\nSample incomplete lexicalisations (first 10):")
    print(
        incomplete_df[["category", "eid", "lex_id", "lex_text",
                        "n_yes_questions", "n_no_questions", "n_extractive_questions",
                        "missing"]]
        .head(10)
        .to_string(index=False)
    )

/tmp/ipykernel_254518/3872436533.py:10: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_PATH)


COVERAGE REPORT — grouped by (category, eid, lex_id)
  Total lexicalisations:    8825
  ✅ Complete:              8594  (97.4%)
  ❌ Incomplete:            231  (2.6%)

Missing breakdown (lexicalisations missing each type):
  • No YES question:        2
  • No NO question:         228
  • <2 extractive:          6

Missing combination breakdown:
                                                   missing  count
                           extractive_questions(2_missing)      2
                                               no_question    224
              no_question; extractive_questions(2_missing)      3
                                              yes_question      1
yes_question; no_question; extractive_questions(2_missing)      1

Full report saved to: coverage_report.csv

Sample incomplete lexicalisations (first 10):
category   eid  lex_id                                                                                                            lex_text  n_yes_questions  n_no_questi

In [22]:
import pandas as pd

# ─────────────────────────────────────────────────────────────
# CONFIG — can run standalone or after the previous script
# ─────────────────────────────────────────────────────────────
INPUT_PATH       = "webnlg_qa_merged_validated.csv"
OUTPUT_PATH_FULL = "entity_coverage_report.csv"

df = pd.read_csv(INPUT_PATH)
df["effective_answer"] = df["corrected_answer"].fillna(df["answer"]).astype(str).str.strip()

# ─────────────────────────────────────────────────────────────
# GROUP BY (category, eid) — aggregate ACROSS all lex_ids
# ─────────────────────────────────────────────────────────────
GROUP_KEYS = ["category", "eid"]

records = []

for group_key, group in df.groupby(GROUP_KEYS):
    cat, eid = group_key

    yes_no_rows = group[group["question_type"] == "yes_no"]
    extr_rows   = group[group["question_type"] == "extractive"]

    # Distinct questions across ALL lexicalisations of this entity
    yes_questions  = yes_no_rows[yes_no_rows["effective_answer"].str.lower() == "yes"]["question"].unique().tolist()
    no_questions   = yes_no_rows[yes_no_rows["effective_answer"].str.lower() == "no"]["question"].unique().tolist()
    extr_questions = extr_rows["question"].unique().tolist()

    n_lex = group["lex_id"].nunique()

    has_yes    = len(yes_questions) >= 1
    has_no     = len(no_questions)  >= 1
    has_2_extr = len(extr_questions) >= 2
    is_complete = has_yes and has_no and has_2_extr

    missing = []
    if not has_yes:
        missing.append("yes_question")
    if not has_no:
        missing.append("no_question")
    if not has_2_extr:
        missing.append(f"extractive_questions({2 - len(extr_questions)}_missing)")

    records.append({
        "category":               cat,
        "eid":                    eid,
        "n_lexicalisations":      n_lex,
        "n_yes_questions":        len(yes_questions),
        "n_no_questions":         len(no_questions),
        "n_extractive_questions": len(extr_questions),
        "has_yes":                has_yes,
        "has_no":                 has_no,
        "has_2_extractive":       has_2_extr,
        "is_complete":            is_complete,
        "missing":                "; ".join(missing) if missing else "",
        "yes_questions":          " | ".join(yes_questions),
        "no_questions":           " | ".join(no_questions),
        "extractive_questions":   " | ".join(extr_questions),
    })

report_df = pd.DataFrame(records)

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
total      = len(report_df)
complete   = report_df["is_complete"].sum()
incomplete = total - complete

missing_combo = (
    report_df[~report_df["is_complete"]]
    .groupby("missing")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("=" * 60)
print("ENTITY-LEVEL COVERAGE REPORT — grouped by (category, eid)")
print("=" * 60)
print(f"  Total entities:        {total}")
print(f"  ✅ Complete:           {complete}  ({100*complete/total:.1f}%)")
print(f"  ❌ Incomplete:         {incomplete}  ({100*incomplete/total:.1f}%)")
print()
print("Missing breakdown (entities missing each type):")
print(f"  • No YES question:     {(~report_df['has_yes']).sum()}")
print(f"  • No NO question:      {(~report_df['has_no']).sum()}")
print(f"  • <2 extractive:       {(~report_df['has_2_extractive']).sum()}")
print()
print("Missing combination breakdown:")
print(missing_combo.to_string(index=False))
print("=" * 60)

# Per-category breakdown
print("\nPer-category completeness:")
cat_summary = (
    report_df.groupby("category")
    .agg(
        total_entities=("eid", "count"),
        complete_entities=("is_complete", "sum"),
    )
    .assign(pct_complete=lambda x: (100 * x["complete_entities"] / x["total_entities"]).round(1))
    .sort_values("pct_complete")
)
print(cat_summary.to_string())

# ─────────────────────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────────────────────
report_df.to_csv(OUTPUT_PATH_FULL, index=False)
print(f"\nFull report saved to: {OUTPUT_PATH_FULL}")

# Sample incomplete entities
incomplete_df = report_df[~report_df["is_complete"]]
if len(incomplete_df) > 0:
    print(f"\nSample incomplete entities (first 10):")
    print(
        incomplete_df[["category", "eid", "n_lexicalisations",
                        "n_yes_questions", "n_no_questions", "n_extractive_questions",
                        "missing"]]
        .head(10)
        .to_string(index=False)
    )

/tmp/ipykernel_254518/2297758129.py:9: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_PATH)


ENTITY-LEVEL COVERAGE REPORT — grouped by (category, eid)
  Total entities:        3548
  ✅ Complete:           3253  (91.7%)
  ❌ Incomplete:         295  (8.3%)

Missing breakdown (entities missing each type):
  • No YES question:     15
  • No NO question:      288
  • <2 extractive:       8

Missing combination breakdown:
                                     missing  count
                                 no_question    272
                   yes_question; no_question     13
             extractive_questions(1_missing)      5
no_question; extractive_questions(1_missing)      3
                                yes_question      2

Per-category completeness:
                      total_entities  complete_entities  pct_complete
category                                                             
Politician                       299                255          85.3
MeanOfTransportation             309                273          88.3
SportsTeam                       251                2